In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
silver_mapeamento= {
    'temp_silver_clientes' : f'{silver_path}/clientes/',
    'temp_silver_itens_pedido' : f'{silver_path}/itens_pedido/',
    'temp_silver_pedidos' : f'{silver_path}/pedidos/',
    'temp_silver_produtos' : f'{silver_path}/produtos/',
    'temp_silver_vendedores' : f'{silver_path}/vendedores/'

}
for view_name, path in silver_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )

In [0]:
%sql
select * from temp_silver_vendedores

In [0]:
%sql
select * from temp_silver_pedidos

In [0]:
%sql
select * from temp_silver_produtos

In [0]:
%sql
select * from temp_silver_itens_pedido

In [0]:
%python
ranking_vendedores = spark.sql("""
        select 
            v.nome_vendedor,
            v.regiao,
            COUNT(distinct p.id_pedido) as Total_pedidos,
            ROUND(SUM(pr.preco_unitario * ip.quantidade), 2) as Receita_gerada,
            ROUND(SUM(pr.preco_unitario * ip.quantidade) / COUNT(distinct p.id_pedido), 2) as Ticket_medio
        from temp_silver_vendedores v
        left join temp_silver_pedidos p on v.id_vendedor = p.id_vendedor
        left join temp_silver_itens_pedido ip on p.id_pedido = ip.id_pedido
        left join temp_silver_produtos pr on ip.id_produto = pr.id_produto

        group by v.nome_vendedor, v.regiao

        order by Receita_gerada desc 

                               
                               """)

# salvar em Delta na Gold
ranking_vendedores.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeShema','true')\
                .save(f'{gold_path}/ranking_vendedores/')

In [0]:
%sql
create table if not exists workspace.techvenda.ranking_vendedores
select * from delta. `/Volumes/workspace/techvenda/filestore/gold/ranking_vendedores/`

In [0]:
%sql
select * from workspace.techvenda.ranking_vendedores